# 11 — Re-clasificación zero-shot con etiquetas refinadas (v2)

Re-ejecuta BART-MNLI sobre el corpus con **etiquetas más específicas** para resolver el solapamiento semántico detectado en la v1 entre `phishing_identity` y `gov_impersonation`.

## Cambios clave respecto al notebook 09

1. **Etiquetas refinadas** — más descriptivas y semánticamente excluyentes.
2. **NO sobrescribe** el CSV del run anterior. Guarda en `scam_us_FINAL_classified_v2.csv`.
3. **Checkpoints más frecuentes** (cada 50 tweets en vez de 100) — más seguridad.
4. **Carga BERTopic ya hecho** del CSV anterior — no se reejecuta, solo reclasifica.
5. **Estimación de tiempo realista**: ~5-7 horas reales en CPU Intel.

## Antes de ejecutar

- ✅ Mac enchufado.
- ✅ `caffeinate -dimsu` corriendo en otra terminal.
- ✅ Kernel: **Python (tfm-nlp)**.
- ✅ Backup del `scam_us_FINAL_classified.csv` anterior en Drive (por si acaso).


## Verificación de entorno

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy, torch, transformers, sentence_transformers, pandas
print(f"numpy:                 {numpy.__version__}")
print(f"torch:                 {torch.__version__}")
print(f"transformers:          {transformers.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"pandas:                {pandas.__version__}")
print()
print("✓ Entorno listo")


## Carga del corpus

Cargamos el corpus que ya tiene BERTopic procesado, **NO lo reejecutamos**. Solo reclasificaremos con zero-shot v2.


In [ ]:
import pandas as pd
import numpy as np
import os
import re
import time

pd.set_option('display.max_colwidth', None)

# Cargamos el CSV que ya tiene BERTopic (más eficiente que partir del v8)
INPUT_CSV = "../data/raw/scam_us_v8_with_bertopic.csv"

if not os.path.exists(INPUT_CSV):
    # Si no existe el de BERTopic intermedio, usamos el v8 limpio
    INPUT_CSV = "../data/raw/scam_us_FINAL_v8.csv"
    print(f"⚠️ No se encontró el CSV de BERTopic. Usando {INPUT_CSV}")
    print(f"   (no tendrás columnas bertopic_id ni bertopic_keywords)")

df = pd.read_csv(INPUT_CSV)
print(f"Tweets cargados: {len(df)}")
print(f"Columnas: {list(df.columns)[:10]}...")

# Asegurar clean_text
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_analysis(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

if "clean_text" not in df.columns or df["clean_text"].isna().all():
    df["clean_text"] = df["text"].apply(clean_for_analysis)
else:
    df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x if x else "")
    mask_empty = df["clean_text"] == ""
    df.loc[mask_empty, "clean_text"] = df.loc[mask_empty, "text"].apply(clean_for_analysis)

df = df[df["clean_text"].str.len() >= 20].reset_index(drop=True)
print(f"Tras filtrar textos cortos: {len(df)}")


## Etiquetas refinadas (v2)

**Cambios clave:**
- `phishing` ahora especifica "private company" para distinguirlo de gov.
- `gov_impersonation` cita IRS, USPS, FedEx, toll → palabras que el modelo reconoce.
- Resto: descripciones más naturales y específicas.


In [ ]:
CANDIDATE_LABELS_V2 = [
    "investment scam or cryptocurrency trading fraud",
    "romance scam or online dating fraud",
    "phishing email or scam text impersonating a private company or bank",
    "impersonation of government agency such as IRS USPS FedEx or toll authority",
    "bank fraud or unauthorized wire transfer",
    "scam involving Zelle Venmo Cash App PayPal or Apple Pay payment app",
    "Ponzi scheme or pyramid scheme",
    "fake tech support call or computer pop-up scam",
    "fake job offer or employment scam",
    "fake charity or donation request scam",
    "insurance fraud or insurance claim manipulation",
    "corporate fraud or securities market manipulation",
    "tax evasion or tax preparation fraud",
    "not related to financial fraud at all",
]

LABEL_TO_CODE_V2 = {
    "investment scam or cryptocurrency trading fraud": "investment_crypto",
    "romance scam or online dating fraud": "romance",
    "phishing email or scam text impersonating a private company or bank": "phishing_identity",
    "impersonation of government agency such as IRS USPS FedEx or toll authority": "gov_impersonation",
    "bank fraud or unauthorized wire transfer": "bank_wire",
    "scam involving Zelle Venmo Cash App PayPal or Apple Pay payment app": "payment_app",
    "Ponzi scheme or pyramid scheme": "ponzi_pyramid",
    "fake tech support call or computer pop-up scam": "tech_support",
    "fake job offer or employment scam": "employment",
    "fake charity or donation request scam": "charity",
    "insurance fraud or insurance claim manipulation": "insurance",
    "corporate fraud or securities market manipulation": "corporate",
    "tax evasion or tax preparation fraud": "tax",
    "not related to financial fraud at all": "not_related",
}

print(f"Categorías candidatas: {len(CANDIDATE_LABELS_V2)}\n")
for i, lbl in enumerate(CANDIDATE_LABELS_V2, 1):
    print(f"  {i:2d}. {lbl}")


## Carga de BART-MNLI

El modelo ya está descargado de la ejecución anterior, así que la carga será rápida (1-2 min).


In [ ]:
print("⏳ Cargando modelo BART-MNLI (ya en caché)...")
t0 = time.time()

from transformers import pipeline
import torch

device = -1  # CPU
print(f"   Dispositivo: CPU")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

t_load = time.time() - t0
print(f"✓ Modelo cargado en {t_load/60:.1f} min")


## Re-clasificación con etiquetas v2

⏳ **Tarda 5-7 horas en CPU Intel.** Checkpoints cada 50 tweets.

NO se sobrescribe el CSV anterior — guardamos en `scam_us_FINAL_classified_v2.csv` y `scam_us_v8_classification_v2_checkpoint.csv`.


In [ ]:
texts_to_classify = df["clean_text"].tolist()
n = len(texts_to_classify)
BATCH_SIZE = 8
CHECKPOINT_EVERY = 50      # más frecuente que el run anterior

predictions = []
scores = []

print(f"⏳ Clasificando {n} tweets en {len(CANDIDATE_LABELS_V2)} categorías (v2)...")
print(f"   Lote: {BATCH_SIZE} | Checkpoint cada {CHECKPOINT_EVERY}")
print(f"   Output:           ../data/raw/scam_us_FINAL_classified_v2.csv")
print(f"   Checkpoints:      ../data/raw/scam_us_v8_classification_v2_checkpoint.csv")
print(f"   ESTIMACIÓN: 5-7 HORAS en CPU Intel. Sé paciente.\n")

t0 = time.time()
last_print = t0

for i in range(0, n, BATCH_SIZE):
    batch = texts_to_classify[i:i+BATCH_SIZE]

    try:
        results = classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS_V2,
            multi_label=False,
        )
        if isinstance(results, dict):
            results = [results]
    except Exception as e:
        print(f"  ⚠️ Error en lote {i}: {e}")
        for _ in batch:
            predictions.append("not related to financial fraud at all")
            scores.append(0.0)
        continue

    for r in results:
        predictions.append(r["labels"][0])
        scores.append(r["scores"][0])

    if (time.time() - last_print) > 30 or (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - t0
        done = min(i + BATCH_SIZE, n)
        eta = (elapsed / done) * (n - done) if done > 0 else 0
        print(f"  {done}/{n} ({done/n*100:.1f}%) — {elapsed/60:.1f} min — ETA: {eta/60:.1f} min")
        last_print = time.time()

    # Checkpoint cada 50
    if (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0 and (i + BATCH_SIZE) < n:
        df_partial = df.iloc[:len(predictions)].copy()
        df_partial["predicted_label_v2"] = predictions
        df_partial["confidence_score_v2"] = scores
        df_partial.to_csv("../data/raw/scam_us_v8_classification_v2_checkpoint.csv", index=False)

t_classify = time.time() - t0
print(f"\n✓ Clasificación v2 completada en {t_classify/60:.1f} min")


In [ ]:
# Añadir predicciones v2 al DataFrame
df["predicted_label_v2"] = predictions[:len(df)]
df["confidence_score_v2"] = scores[:len(df)]
df["predicted_category_v2"] = df["predicted_label_v2"].map(LABEL_TO_CODE_V2)
df["is_relevant_v2"] = df["predicted_category_v2"] != "not_related"

# Distribución
print("=== DISTRIBUCIÓN DE CATEGORÍAS V2 (REFINADAS) ===\n")
counts = df["predicted_category_v2"].value_counts()
total = len(df)
for cat, n_cat in counts.items():
    pct = n_cat / total * 100
    print(f"  {cat:<22} {n_cat:>5}  ({pct:>5.1f}%)")

print(f"\nRelevantes: {df['is_relevant_v2'].sum()} / {total} ({df['is_relevant_v2'].mean()*100:.1f}%)")


In [ ]:
# Confianza
print("=== CONFIANZA V2 ===\n")
print(f"  Media:    {df['confidence_score_v2'].mean():.3f}")
print(f"  Mediana:  {df['confidence_score_v2'].median():.3f}")
print(f"  Min:      {df['confidence_score_v2'].min():.3f}")
print(f"  Max:      {df['confidence_score_v2'].max():.3f}")
print(f"\n  Alta confianza (≥0.7):  {(df['confidence_score_v2'] >= 0.7).sum()}")
print(f"  Media (0.4-0.7):        {((df['confidence_score_v2'] >= 0.4) & (df['confidence_score_v2'] < 0.7)).sum()}")
print(f"  Baja (<0.4):            {(df['confidence_score_v2'] < 0.4).sum()}")


## Comparativa v1 vs v2

Si el CSV del run anterior existe, comparamos cuánto cambia la distribución.


In [ ]:
V1_PATH = "../data/raw/scam_us_FINAL_classified.csv"
if os.path.exists(V1_PATH):
    df_v1 = pd.read_csv(V1_PATH)
    if "predicted_category" in df_v1.columns:
        # Mergear por tweet_id
        df_compare = df.merge(
            df_v1[["tweet_id", "predicted_category"]].rename(
                columns={"predicted_category": "predicted_category_v1"}
            ),
            on="tweet_id",
            how="left",
        )
        
        print("=== COMPARATIVA V1 vs V2 (en mismos tweets) ===\n")
        # Distribución lado a lado
        v1_counts = df_compare["predicted_category_v1"].value_counts()
        v2_counts = df_compare["predicted_category_v2"].value_counts()
        all_cats = sorted(set(v1_counts.index) | set(v2_counts.index))
        print(f"{'categoría':<22} {'v1':>6} {'v2':>6} {'cambio':>8}")
        print("-" * 50)
        for cat in all_cats:
            n1 = v1_counts.get(cat, 0)
            n2 = v2_counts.get(cat, 0)
            diff = n2 - n1
            sign = "+" if diff > 0 else ""
            print(f"  {cat:<20} {n1:>6} {n2:>6} {sign}{diff:>6}")
        
        # Cuántos tweets cambiaron de categoría
        n_changed = (df_compare["predicted_category_v1"] != df_compare["predicted_category_v2"]).sum()
        print(f"\nTweets que cambiaron de categoría: {n_changed} ({n_changed/len(df_compare)*100:.1f}%)")
    else:
        print("V1 no tiene columna predicted_category")
else:
    print("No existe el CSV v1 para comparar")


## Guardado del CSV v2

NO sobrescribimos el v1. El v2 va a un archivo separado.


In [ ]:
OUTPUT_V2 = "../data/raw/scam_us_FINAL_classified_v2.csv"
df.to_csv(OUTPUT_V2, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT_V2}")
print(f"  Total filas: {len(df)}")
print()
print("📦 Archivos:")
print(f"  scam_us_FINAL_classified.csv      ← run v1 (original, intacto)")
print(f"  scam_us_FINAL_classified_v2.csv   ← run v2 (este, refinado)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del v2 a Drive.")


## Ejemplos de la nueva clasificación

Verifica que ahora `gov_impersonation` recibe más tweets de IRS/USPS/toll/FedEx.


In [ ]:
print("=== 3 EJEMPLOS DE CADA CATEGORÍA V2 (alta confianza) ===\n")
for cat_code in df["predicted_category_v2"].value_counts().index:
    subset = df[
        (df["predicted_category_v2"] == cat_code) &
        (df["confidence_score_v2"] >= 0.5)
    ].nlargest(3, "confidence_score_v2")
    n_total = (df["predicted_category_v2"] == cat_code).sum()
    print(f"--- {cat_code} ({n_total} total) ---")
    for _, row in subset.iterrows():
        print(f"  [{row['confidence_score_v2']:.2f}] {str(row['text'])[:200]}")
    print()


In [ ]:
# Específicamente: ver ejemplos de gov_impersonation
print("=== EJEMPLOS DE gov_impersonation EN V2 ===\n")
gov_subset = df[df["predicted_category_v2"] == "gov_impersonation"].nlargest(
    10, "confidence_score_v2"
)
for _, row in gov_subset.iterrows():
    print(f"  [{row['confidence_score_v2']:.2f}] @{row['username']}")
    print(f"    {str(row['text'])[:250]}")
    print()
